# Clasificador LSM — entrenamiento y comparación de modelos

**SeñasUTSCMX** · Adrián Gottfried · UTSC 2026

Este notebook entrena un clasificador de las letras estáticas del abecedario de la
Lengua de Señas Mexicana a partir de los 21 landmarks que MediaPipe extrae de la mano.

El flujo es:

1. Cargar y explorar los datos
2. Construir las representaciones (features) y compararlas
3. Partir los datos **por sesión** para evitar fuga de información
4. Entrenar y comparar k-NN, Random Forest, SVM y MLP
5. Evaluar el ganador a fondo (matriz de confusión, accuracy por letra)
6. Guardar el modelo para integrarlo en `ml-service`

Ejecuta las celdas en orden. Cada una explica qué hace y por qué.

## 1. Cargar y explorar

Primero miramos qué tenemos: cuántas muestras, cuántas letras, si hay datos faltantes
y de cuántas sesiones de grabación vienen. Ese último dato es el que decide cómo
vamos a evaluar el modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/landmarks_raw.csv')

print(f'Muestras totales : {len(df)}')
print(f'Letras distintas : {df["label"].nunique()}')
print(f'Sujetos          : {df["sujeto"].unique().tolist()}')
print(f'Sesiones         : {df["sesion"].nunique()}')
print(f'Valores nulos    : {df.isna().sum().sum()}')
df.head(3)

### Distribución por letra

Un dataset desbalanceado hace que el modelo favorezca las clases con más ejemplos.
Queremos ver barras parejas.

In [ ]:
conteo = df['label'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
conteo.plot(kind='bar', ax=ax, color='#FF8300')
ax.axhline(conteo.mean(), color='#0C7669', ls='--', label=f'media {conteo.mean():.0f}')
ax.set_title('Muestras por letra')
ax.set_xlabel('')
ax.legend()
plt.tight_layout(); plt.show()

print(f'mínimo {conteo.min()} | máximo {conteo.max()} | desbalance {conteo.max()/conteo.min():.2f}x')

### Sesiones por letra

Esto es lo importante para evaluar bien. Si una letra se grabó en **una sola sesión**,
no podemos ponerla en train y test a la vez sin hacer trampa.

Si ves muchas letras con 1 sola sesión, la evaluación honesta será más difícil y
conviene grabar otra tanda en otro momento (otra luz, otra ropa, otro día).

In [ ]:
ses = df.groupby('label')['sesion'].nunique().sort_values()
print(ses)
print()
print(f'Letras con una sola sesión: {(ses == 1).sum()} de {len(ses)}')
print(f'Sesiones totales: {df["sesion"].nunique()}')

## 2. Construir las features

MediaPipe entrega **dos** juegos de 21 puntos:

| Juego | Qué es | Ventaja |
|---|---|---|
| `img_*` | Coordenadas 0–1 relativas al cuadro de la cámara | Refleja dónde está la mano en pantalla |
| `wld_*` | Metros, con el origen en la muñeca | Ya es invariante a posición y a escala |

Guardamos ambos crudos justamente para poder comparar. Vamos a probar cuatro
representaciones y dejar que los números decidan:

- **A · img crudo** — la línea base ingenua
- **B · img normalizado** — centrado en la muñeca y escalado por el tamaño de la palma
- **C · world crudo** — lo que MediaPipe ya da normalizado
- **D · world + distancias** — world más las distancias entre yemas, que capturan
  explícitamente lo que distingue pares difíciles como U/V o F/W

In [ ]:
IMG = [f'img_{i}_{c}' for i in range(21) for c in 'xyz']
WLD = [f'wld_{i}_{c}' for i in range(21) for c in 'xyz']

def a_puntos(X):
    """(n, 63) -> (n, 21, 3) para operar por punto."""
    return X.reshape(len(X), 21, 3)

def normalizar(X):
    """Centra en la muñeca (punto 0) y escala por el tamaño de la palma.

    Sin esto, mover la mano por el cuadro o acercarla cambia todos los números
    aunque la seña sea la misma.
    """
    P = a_puntos(X).copy()
    P -= P[:, 0:1, :]                       # origen en la muñeca
    escala = np.linalg.norm(P[:, 9, :], axis=1, keepdims=True)   # muñeca -> nudillo medio
    escala[escala == 0] = 1e-8
    P /= escala[:, :, None]
    return P.reshape(len(P), -1)

def distancias_yemas(X):
    """Distancias entre las 5 yemas + de cada yema a la muñeca (15 + 5 = 20 valores).

    Le da al modelo, ya masticado, lo que separa U de V (yemas juntas o separadas)
    o F de W. Un modelo puede deducirlo solo, pero dárselo explícito ayuda.
    """
    P = a_puntos(X)
    yemas = P[:, [4, 8, 12, 16, 20], :]
    out = []
    for i in range(5):
        for j in range(i + 1, 5):
            out.append(np.linalg.norm(yemas[:, i] - yemas[:, j], axis=1))
    for i in range(5):
        out.append(np.linalg.norm(yemas[:, i] - P[:, 0], axis=1))
    return np.stack(out, axis=1)

X_img = df[IMG].to_numpy(dtype=np.float32)
X_wld = df[WLD].to_numpy(dtype=np.float32)
y     = df['label'].to_numpy()
grupos = df['sesion'].to_numpy()

REPRESENTACIONES = {
    'A · img crudo'          : X_img,
    'B · img normalizado'    : normalizar(X_img),
    'C · world crudo'        : X_wld,
    'D · world + distancias' : np.hstack([normalizar(X_wld), distancias_yemas(X_wld)]),
}

for nombre, X in REPRESENTACIONES.items():
    print(f'{nombre:24s} -> {X.shape[1]:3d} features')

## 3. La partición: por sesión, no aleatoria

**Esta es la celda más importante del notebook.**

Las muestras se capturaron a 10 por segundo, así que dos consecutivas están separadas
por 100 ms y son casi idénticas. Si partimos al azar, un frame y su gemelo caen uno en
train y otro en test: el modelo 'acierta' porque ya vio la respuesta. El accuracy sale
inflado y luego la app falla en la vida real.

La partición honesta agrupa **por sesión de grabación**: una sesión entera va a train o
a test, nunca partida. Así medimos si el modelo generaliza a grabaciones nuevas.

Abajo calculamos las dos para que veas la diferencia con tus propios datos.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_demo = REPRESENTACIONES['D · world + distancias']
modelo_demo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

# --- partición ingenua (aleatoria) ---
Xtr, Xte, ytr, yte = train_test_split(X_demo, y, test_size=0.25,
                                      random_state=42, stratify=y)
acc_ingenua = modelo_demo.fit(Xtr, ytr).score(Xte, yte)

# --- partición honesta (por sesión) ---
if df['sesion'].nunique() > 1:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    itr, ite = next(gss.split(X_demo, y, groups=grupos))
    acc_honesta = modelo_demo.fit(X_demo[itr], y[itr]).score(X_demo[ite], y[ite])
else:
    acc_honesta = None

print(f'Partición ALEATORIA (inflada) : {acc_ingenua:.4f}')
if acc_honesta is not None:
    print(f'Partición POR SESIÓN (honesta): {acc_honesta:.4f}')
    print(f'\nDiferencia: {(acc_ingenua - acc_honesta)*100:.1f} puntos de accuracy fantasma.')
    print('Ese hueco es exactamente la fuga de información. El segundo número es el real.')
else:
    print('\nSolo hay UNA sesión de grabación, así que no se puede partir por sesión.')
    print('El número de arriba está inflado y no es fiable.')
    print('RECOMENDACIÓN: graba otra tanda en otro momento (otra luz, otro día) y vuelve aquí.')

## 4. Comparar representaciones y modelos

Ahora sí: entrenamos cuatro familias de modelos sobre las cuatro representaciones y
comparamos. Usamos validación cruzada agrupada por sesión, que repite la partición
honesta varias veces y promedia — más fiable que una sola división.

Los candidatos, de simple a complejo:

| Modelo | Cómo funciona | Por qué está aquí |
|---|---|---|
| **k-NN** | Busca las muestras más parecidas y vota | Línea base. Si algo no le gana, ese algo sobra |
| **Random Forest** | Muchos árboles de decisión votando | Robusto, no necesita escalado, dice qué features importan |
| **SVM (RBF)** | Traza fronteras curvas entre clases | Fuerte con pocos datos y muchas dimensiones |
| **MLP** | Red neuronal pequeña | Suele ganar, y es el más vistoso de defender |

Esta celda tarda unos minutos. Es la que hace el trabajo.

In [ ]:
from sklearn.model_selection import cross_val_score, GroupKFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

MODELOS = {
    'k-NN'          : lambda: make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=5, weights='distance')),
    'Random Forest' : lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'SVM (RBF)'     : lambda: make_pipeline(StandardScaler(),
                          SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)),
    'MLP'           : lambda: make_pipeline(StandardScaler(),
                          MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=600,
                                        early_stopping=True, random_state=42)),
}

n_ses = df['sesion'].nunique()
if n_ses > 1:
    cv = GroupKFold(n_splits=min(5, n_ses))
    grupos_cv = grupos
    print(f'Validación cruzada agrupada por sesión ({cv.get_n_splits()} particiones).\n')
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grupos_cv = None
    print('AVISO: una sola sesión -> CV estratificada. Los números saldrán optimistas.\n')

filas = []
for nombre_rep, X in REPRESENTACIONES.items():
    for nombre_mod, constructor in MODELOS.items():
        scores = cross_val_score(constructor(), X, y, cv=cv, groups=grupos_cv, n_jobs=-1)
        filas.append({'representación': nombre_rep, 'modelo': nombre_mod,
                      'accuracy': scores.mean(), 'desv': scores.std()})
        print(f'{nombre_rep:24s} | {nombre_mod:14s} | {scores.mean():.4f} ± {scores.std():.4f}')

res = pd.DataFrame(filas).sort_values('accuracy', ascending=False)
print('\n\nRanking completo:')
res.reset_index(drop=True)

In [ ]:
tabla = res.pivot(index='modelo', columns='representación', values='accuracy')

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(tabla, annot=True, fmt='.3f', cmap='YlOrBr', ax=ax,
            cbar_kws={'label': 'accuracy'})
ax.set_title('Accuracy por modelo y representación')
plt.tight_layout(); plt.show()

mejor = res.iloc[0]
print(f'Ganador: {mejor["modelo"]} sobre "{mejor["representación"]}"'
      f' -> {mejor["accuracy"]:.4f}')

## 5. Evaluar al ganador a fondo

El accuracy global esconde información. Lo que de verdad importa es **qué letras**
falla y **con cuáles las confunde**. Ahí es donde esperamos ver los pares difíciles:
U/V, F/W, M/N.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

X_mejor = REPRESENTACIONES[mejor['representación']]
clf = MODELOS[mejor['modelo']]()

if n_ses > 1:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
    itr, ite = next(gss.split(X_mejor, y, groups=grupos))
else:
    itr, ite = train_test_split(np.arange(len(y)), test_size=0.25,
                                random_state=7, stratify=y)

clf.fit(X_mejor[itr], y[itr])
pred = clf.predict(X_mejor[ite])

print(classification_report(y[ite], pred, digits=3))

In [ ]:
letras = sorted(np.unique(y))
cm = confusion_matrix(y[ite], pred, labels=letras)
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_pct, annot=cm, fmt='d', cmap='YlOrBr',
            xticklabels=letras, yticklabels=letras, ax=ax, cbar=False)
ax.set_xlabel('predicho'); ax.set_ylabel('real')
ax.set_title(f'Matriz de confusión — {mejor["modelo"]}')
plt.tight_layout(); plt.show()

### Las confusiones más frecuentes

Esta lista te dice exactamente qué señas necesitan más datos o mejor grabación.
Si un par aparece arriba, vuelve a grabarlo **exagerando la diferencia** entre ambas.

In [ ]:
errores = []
for i, real in enumerate(letras):
    for j, pred_l in enumerate(letras):
        if i != j and cm[i, j] > 0:
            errores.append({'real': real, 'predicho': pred_l, 'veces': cm[i, j],
                            'pct': cm[i, j] / cm[i].sum() * 100})

if errores:
    err = pd.DataFrame(errores).sort_values('veces', ascending=False)
    print('Top confusiones:')
    print(err.head(15).to_string(index=False))
else:
    print('Sin errores en el conjunto de prueba.')
    print('Ojo: si esto pasa con una sola sesión, es señal de fuga, no de perfección.')

## 6. Guardar el modelo

Entrenamos el ganador con **todos** los datos (ya no necesitamos apartar test) y lo
guardamos junto con los metadatos que `ml-service` necesitará para usarlo igual que
aquí: qué representación espera y en qué orden vienen las letras.

In [ ]:
import joblib, json, datetime
from pathlib import Path

Path('modelo').mkdir(exist_ok=True)

final = MODELOS[mejor['modelo']]()
final.fit(X_mejor, y)

joblib.dump(final, 'modelo/clasificador_lsm.joblib')

meta = {
    'creado'          : datetime.datetime.now().isoformat(timespec='seconds'),
    'modelo'          : mejor['modelo'],
    'representacion'  : mejor['representación'],
    'accuracy_cv'     : round(float(mejor['accuracy']), 4),
    'letras'          : sorted(np.unique(y).tolist()),
    'n_muestras'      : int(len(df)),
    'n_sesiones'      : int(df['sesion'].nunique()),
    'sujetos'         : df['sujeto'].unique().tolist(),
    'n_features'      : int(X_mejor.shape[1]),
}

with open('modelo/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('Guardado en modelo/')
print(json.dumps(meta, indent=2, ensure_ascii=False))

---

## Qué hacer con estos resultados

**Si el accuracy honesto es alto (>90%)** — listo para integrar en `ml-service`.

**Si es bajo o hay confusiones marcadas** — no toques el modelo todavía, mejora los
datos: vuelve a grabar las letras que se confunden, en otra sesión, exagerando la
diferencia entre ellas. Casi siempre rinde más que cambiar de algoritmo.

**Si solo tienes una sesión de grabación** — el número que ves está inflado y no
sabes cuál es el real. Graba otra tanda en otro momento antes de confiar en él.

### Para la documentación de tu tesis

Guarda de aquí: la tabla comparativa de modelos, la matriz de confusión, el accuracy
por letra, y la justificación de la partición por sesión. Eso último es lo que
distingue una evaluación rigurosa de una que se engaña sola.